# Pancancer enrichment analysis step 2: Find enriched pathways
# Version 1: Using GSEApy

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP4_DIR = "step4_outputs"
STEP4_FILE = f"enrichment_gseapy_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
STEP4_FILE_PATH = os.path.join(STEP4_DIR, STEP4_FILE)

if not os.path.isdir(STEP4_DIR):
    os.mkdir(STEP4_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# The ranked enrichment analysis service wants ranking scores
# where "bigger is better", such as expression values or
# t scores. We are ranking by adjusted p-values, where smaller
# is better. So, we'll create a column of (1 - adj_p) to use
# for ranking.
data = data.assign(one_minus_adj_p=1 - data["adj_p"])

# Make a column where all increases are +1 and all decreases 
# are -1, because these are ratioed abundances, so we can't 
# compare magnitudes between genes--we can only compare whether 
# there was a change, and whether it was positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [4]:
data.head()

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,one_minus_adj_p,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0000,0.00000,True,A1BG,1.00000,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0000,0.00000,True,A1CF,1.00000,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0000,0.00000,True,A2M,1.00000,1.0
3,ccrcc,"('A4GALT', 'NP_001304967.1')",0.479410,0.1735,0.21431,False,A4GALT,0.78569,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0000,0.00000,True,AAAS,1.00000,1.0


## Run enrichment analysis

In [5]:
# For each cancer, find enriched pathways based on p value for differential expression   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    
    ranked_data = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "one_minus_adj_p"]].\
        set_index("protein_str").\
        rename(columns={"one_minus_adj_p": f"{cancer_type}_one_minus_adj_p"})
    
    # https://gseapy.readthedocs.io/en/latest/gseapy_example.html#4.-GSEA-Example
    # https://gseapy.readthedocs.io/en/latest/run.html
    gs_res = gp.gsea(
        data=ranked_data,
        gene_sets='Reactome_2016',
        cls= './data/P53.cls', # TODO: Fix this
        permutation_type='phenotype',
        permutation_num=10,
        min_size=1, # TODO: Adjust min and max
        max_size=500, 
        outdir=None,
        no_plot=True,
        method='signal_to_noise', # TODO: Research options.
        processes=4,
        seed=0)
    
    cancer_enriched = enriched_pathways.res2d.assign(cancer_type=cancer_type)
    all_enrichments = all_enrichments.append(cancer_enriched)

In [6]:
all_enrichments.head()

,stId,name,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total
0,R-HSA-3108214,SUMOylation of DNA damage response and repair ...,0.005581,0.045443,0.84264,81,81
1,R-HSA-4551638,SUMOylation of chromatin organization proteins,0.004272,0.071045,0.84264,62,62
2,R-HSA-4570464,SUMOylation of RNA binding proteins,0.003514,0.093002,0.84264,51,51
3,R-HSA-4615885,SUMOylation of DNA replication proteins,0.003445,0.095357,0.84264,50,50
4,R-HSA-159234,Transport of Mature mRNAs Derived from Intronl...,0.003238,0.102846,0.84264,47,47


## Summarize enrichment results from all cancers

In [7]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that cancer type is enriched
# - The average of the adjusted p-values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["stId", "name"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["stId"].apply(
    lambda x: all_enrichments[all_enrichments["stId"] == x].shape[0])

avg_fdr = enrichment_summary["stId"].apply(
    lambda x: all_enrichments.loc[all_enrichments["stId"] == x, "entities_fdr"].mean())

avg_overlap_props = enrichment_summary["stId"].apply(
    lambda x: all_enrichments.loc[all_enrichments["stId"] == x, "entities_ratio"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    avg_fdr=avg_fdr,
    avg_overlap_prop=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(
    by=["times_enriched", "avg_fdr"],
    ascending=[False, True])

enrichment_summary = enrichment_summary.reset_index(drop=True)

In [8]:
enrichment_summary.head()

,stId,name,times_enriched,avg_fdr,avg_overlap_prop
0,R-HSA-6798695,Neutrophil degranulation,8,0.587721,0.033072
1,R-HSA-6791226,Major pathway of rRNA processing in the nucleo...,8,0.725419,0.013022
2,R-HSA-72203,Processing of Capped Intron-Containing Pre-mRNA,8,0.725419,0.017638
3,R-HSA-72163,mRNA Splicing - Major Pathway,8,0.725447,0.012746
4,R-HSA-72172,mRNA Splicing,8,0.725763,0.013504


## Visualize pathways with expression change

In [9]:
# Submit the differential expression data for each cancer type to the analysis service, and get the tokens
# The data we'd submitted previously were the p values for the differential expression, but that's not what
# we want to visualize; we want to see whether the expression was up or down.
diff_expr_tokens = {}

for cancer_type in data["cancer_type"].unique():

    # Select data for that cancer type
    omics = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "simplified_change"]].\
        set_index("protein_str").\
        rename(columns={"simplified_change": f"{cancer_type}_simp_change"})

    analysis_token, unneeded_df = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=omics, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    diff_expr_tokens[cancer_type] = analysis_token

In [14]:
# Select pathways to get URLs for
to_viz = enrichment_summary[["stId", "name"]].iloc[0:10, :]

id_list = []
name_list = []
cancer_type_list = []
expr_list = []
url_list = []

for idx in to_viz.index:
    pathway_id = to_viz.loc[idx, "stId"]
    name = to_viz.loc[idx, "name"]
    
    for cancer_type in data["cancer_type"].unique():
         
        # Get the URL
        expr_vals, url = ut.reactome_pathway_overlay(
            pathway=pathway_id,
            analysis_token=diff_expr_tokens[cancer_type],
            open_browser=False)
        
        id_list.append(pathway_id)
        name_list.append(name)
        cancer_type_list.append(cancer_type)
        expr_list.append(expr_vals[0])
        url_list.append(url)

urls_df = pd.DataFrame(
    {
        "pathway_id": id_list,
        "pathway_name": name_list,
        "cancer_type": cancer_type_list,
        "mean_expr": expr_list,
        "url": url_list
    })

In [15]:
# Print the dataframe in such a way that the links are clickable
urls_df.style.format('<a href="{0}">{0}</a>', subset="url")

,pathway_id,pathway_name,cancer_type,mean_expr,url
0,R-HSA-6798695,Neutrophil degranulation,ccrcc,0.232143,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MTJfMjAzNTM%3D
1,R-HSA-6798695,Neutrophil degranulation,colon,-0.092166,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MTVfMjAzNTQ%3D
2,R-HSA-6798695,Neutrophil degranulation,endometrial,0.284444,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MThfMjAzNTU%3D
3,R-HSA-6798695,Neutrophil degranulation,gbm,0.209821,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MjJfMjAzNTY%3D
4,R-HSA-6798695,Neutrophil degranulation,hnscc,0.086580,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MjZfMjAzNTc%3D
5,R-HSA-6798695,Neutrophil degranulation,lscc,-0.390519,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MjlfMjAzNTg%3D
6,R-HSA-6798695,Neutrophil degranulation,luad,-0.356009,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MzNfMjAzNTk%3D
7,R-HSA-6798695,Neutrophil degranulation,ovarian,-0.242630,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MzZfMjAzNjA%3D
8,R-HSA-6791226,Major pathway of rRNA processing in the nucleolus and cytosol,ccrcc,0.687151,https://reactome.org/PathwayBrowser/#/R-HSA-6791226&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MTJfMjAzNTM%3D
9,R-HSA-6791226,Major pathway of rRNA processing in the nucleolus and cytosol,colon,0.646409,https://reactome.org/PathwayBrowser/#/R-HSA-6791226&DTAB=AN&ANALYSIS=MjAyMDA2MDUxNjU3MTVfMjAzNTQ%3D


In [12]:
# import webbrowser
# for url in urls_df["url"][0:8]:
#     webbrowser.open(url)

In [16]:
# Save the results
urls_df.to_csv(STEP4_FILE_PATH, sep='\t')